# 01. Загрузка и проверка данных

Цель ноутбука: загрузить один файл транзакций из `data/raw/`, нормализовать названия колонок, проверить качество данных и собрать дневную витрину спроса.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_CANDIDATES, PROCESSED_DATA_DIR
from src.data_checks import find_raw_data_file, read_transactions, data_quality_summary
from src.features import clean_transactions, build_daily_sales

In [ ]:
raw_path = find_raw_data_file(RAW_DATA_CANDIDATES)
transactions = read_transactions(raw_path)
transactions.head()

In [ ]:
quality = data_quality_summary(transactions)
quality

In [ ]:
clean = clean_transactions(transactions)
daily_sales = build_daily_sales(transactions)

summary = {
    'rows_raw': len(transactions),
    'rows_clean': len(clean),
    'sku_count': clean['stock_code'].nunique(),
    'country_count': clean['country'].nunique(),
    'date_min': clean['date'].min(),
    'date_max': clean['date'].max(),
}
summary

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
daily_sales.to_parquet(PROCESSED_DATA_DIR / 'daily_sales.parquet', index=False)
quality.to_csv(PROCESSED_DATA_DIR / 'data_quality_summary.csv', index=False)
daily_sales.head()

## Что зафиксировать в отчете

- число строк до очистки: `[A]`;
- число строк после очистки: `[B]`;
- период данных: `[C]` - `[D]`;
- основные проблемы качества: `[E]`.